# 02 — Logs Analysis

Interactive analysis of log data collected by `scripts/collect_and_detect.sh`.  
Mirrors `ml/anomaly_detection/logs_isolation_forest.py` step-by-step so each
feature engineering decision can be inspected and tuned before committing to
the pipeline script.

**Prerequisite**: at least one cron run must have completed so that
`ml/data/logs_*.csv` exists.

## 1. Setup

In [ ]:
import json
import pathlib
import re

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

ML_DIR       = pathlib.Path("..").resolve()
DATA_DIR     = ML_DIR / "data"
OUTPUT_DIR   = ML_DIR / "output"
INCIDENTS_FILE = ML_DIR / "data_ingest" / "incidents.json"

plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (14, 4)})
%matplotlib inline

## 2. Load Data

In [ ]:
log_files = sorted(DATA_DIR.glob("logs_*.csv"))
print(f"Found {len(log_files)} log file(s):")
for f in log_files:
    print(f"  {f.name}")

if not log_files:
    raise FileNotFoundError(f"No logs_*.csv in {DATA_DIR}. Run collect_and_detect.sh first.")

# To combine all runs: pd.concat([pd.read_csv(f) for f in log_files])
# Default: use the latest run.
SELECTED_FILE = log_files[-1]
print(f"\nUsing: {SELECTED_FILE.name}")

df_raw = pd.read_csv(SELECTED_FILE)
df_raw["timestamp"] = pd.to_datetime(df_raw["timestamp"], utc=True)
df_raw = df_raw.sort_values("timestamp").reset_index(drop=True)
print(f"Rows: {len(df_raw):,}  |  Columns: {list(df_raw.columns)}")
df_raw.head(3)

## 3. EDA

In [ ]:
# Log volume over time (per-minute buckets)
volume = df_raw.set_index("timestamp").resample("1min").size().rename("log_count")
fig, ax = plt.subplots()
volume.plot(ax=ax, color="steelblue")
ax.set(title="Log volume per minute", xlabel="", ylabel="count")
plt.tight_layout()
plt.show()

In [ ]:
# Log level distribution
if "level" in df_raw.columns:
    level_counts = df_raw["level"].str.lower().value_counts()
    fig, ax = plt.subplots(figsize=(6, 3))
    level_counts.plot.bar(ax=ax, color="steelblue", edgecolor="white")
    ax.set(title="Log level distribution", xlabel="level", ylabel="count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("No 'level' column — skipping level distribution.")

In [ ]:
# Top-N message prefixes
msg_col = next((c for c in ["message", "msg", "raw"] if c in df_raw.columns), None)
if msg_col:
    top_msgs = (
        df_raw[msg_col].astype(str)
        .str[:60]                     # first 60 chars as the key
        .value_counts()
        .head(10)
    )
    print("Top-10 message prefixes:")
    print(top_msgs.to_string())
else:
    print("No message column found.")

In [ ]:
# Sample error-level lines
if "level" in df_raw.columns and msg_col:
    errors = df_raw[df_raw["level"].str.lower() == "error"][["timestamp", msg_col]]
    print(f"{len(errors)} error lines. Sample:")
    display(errors.head(5))
else:
    print("Cannot filter by level (column missing).")

## 4. Feature Engineering

In [ ]:
# ── Per-line features (mirrors logs_isolation_forest.py) ─────────────────────
ERROR_KW  = ["error","exception","traceback","fatal","critical","fail","failed","failure","crash","panic"]
WARN_KW   = ["warn","warning","deprecated","timeout","retry","slow"]
LEVEL_MAP = {"debug": 0, "info": 1, "warning": 2, "warn": 2, "error": 3, "critical": 4}

df = df_raw.copy()

# level numeric
if "level" in df.columns:
    df["level_numeric"] = df["level"].str.lower().map(LEVEL_MAP).fillna(1).astype(int)
else:
    df["level_numeric"] = 1

# message features
raw = df[msg_col].astype(str).str.lower() if msg_col else pd.Series("", index=df.index)
df["msg_length"]          = raw.str.len()
df["error_keyword_count"] = sum(raw.str.count(kw) for kw in ERROR_KW)
df["warn_keyword_count"]  = sum(raw.str.count(kw) for kw in WARN_KW)
df["has_traceback"]       = raw.str.contains("traceback|exception", regex=True).astype(int)
df["has_timeout"]         = raw.str.contains("timeout").astype(int)
df["numeric_count"]       = raw.str.count(r"\d+")

print("Per-line features added:")
df[["level_numeric","msg_length","error_keyword_count","warn_keyword_count",
    "has_traceback","has_timeout","numeric_count"]].describe()

In [ ]:
# ── Window aggregation — compare 1 min, 5 min, 10 min ────────────────────────
def aggregate_windows(df, freq):
    g = df.set_index("timestamp").resample(freq)
    eps = 1e-6
    w = pd.DataFrame({
        "log_count":         g.size(),
        "mean_level":        g["level_numeric"].mean(),
        "max_level":         g["level_numeric"].max(),
        "total_msg_length":  g["msg_length"].sum(),
        "error_kw_sum":      g["error_keyword_count"].sum(),
        "warn_kw_sum":       g["warn_keyword_count"].sum(),
        "traceback_count":   g["has_traceback"].sum(),
        "timeout_count":     g["has_timeout"].sum(),
    }).fillna(0)
    w["error_rate_in_window"] = w["error_kw_sum"] / (w["log_count"] + eps)
    w["log_count_delta"]      = w["log_count"].diff().fillna(0)
    roll5 = w["log_count"].rolling(5, min_periods=1)
    w["log_count_roll_mean"]  = roll5.mean()
    w["log_count_roll_std"]   = roll5.std().fillna(0)
    denom = (w["log_count_roll_std"] + eps)
    w["log_burst_z"]          = (w["log_count"] - w["log_count_roll_mean"]) / denom
    return w

windows_1m  = aggregate_windows(df, "1min")
windows_5m  = aggregate_windows(df, "5min")
windows_10m = aggregate_windows(df, "10min")

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=False)
for ax, (w, label) in zip(axes, [(windows_1m, "1 min"), (windows_5m, "5 min"), (windows_10m, "10 min")]):
    ax.plot(w.index, w["log_count"],   label="log_count",   color="steelblue")
    ax.plot(w.index, w["error_kw_sum"],label="error_kw_sum",color="tomato")
    ax.set_title(f"Window: {label}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Choose a window size based on the plots above.
Set `WINDOW` below before continuing.

In [ ]:
WINDOW = "1min"   # ← adjust here
windowed = aggregate_windows(df, WINDOW)
print(f"Windowed shape: {windowed.shape}")
windowed.head(3)

## 5. Anomaly Scoring

In [ ]:
# Set train-end to match the run (70 % of the window by default)
t_start = windowed.index.min()
t_end   = windowed.index.max()
TRAIN_END = t_start + (t_end - t_start) * 0.7
print(f"Train : {t_start}  →  {TRAIN_END}")
print(f"Test  : {TRAIN_END}  →  {t_end}")

FEATURE_COLS = [
    "log_count", "mean_level", "max_level", "error_kw_sum",
    "warn_kw_sum", "traceback_count", "timeout_count",
    "error_rate_in_window", "log_count_delta", "log_burst_z",
]

X = windowed[FEATURE_COLS].fillna(0).values
mask_train = windowed.index <= TRAIN_END

scaler = StandardScaler()
X_train = scaler.fit_transform(X[mask_train])
X_all   = scaler.transform(X)

CONTAMINATION = 0.02   # ← tune here
clf = IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=42, n_jobs=-1)
clf.fit(X_train)

windowed = windowed.copy()
windowed["anomaly_score"] = clf.decision_function(X_all)
windowed["is_anomaly"]    = (clf.predict(X_all) == -1).astype(int)
windowed["in_train"]      = mask_train.astype(int)

n = windowed["is_anomaly"].sum()
print(f"Anomalies flagged: {n} / {len(windowed)}  ({100*n/len(windowed):.1f} %)")

In [ ]:
# 3-panel plot: log_count | error_kw_sum | anomaly_score
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
anom_idx  = windowed[windowed["is_anomaly"] == 1].index

for ax, col, color in zip(axes,
    ["log_count", "error_kw_sum", "anomaly_score"],
    ["steelblue", "tomato", "purple"]):
    ax.plot(windowed.index, windowed[col], color=color, linewidth=0.9)
    ax.scatter(anom_idx, windowed.loc[anom_idx, col], color="red", s=20, zorder=5, label="anomaly")
    ax.set_ylabel(col)
    ax.legend(fontsize=8, loc="upper right")

axes[0].set_title("Log anomaly detection — IsolationForest")
axes[2].axhline(0, color="gray", linestyle="--", linewidth=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Contamination sensitivity: try a few values
results = {}
for c in [0.01, 0.02, 0.05, 0.10]:
    m = IsolationForest(n_estimators=200, contamination=c, random_state=42, n_jobs=-1)
    m.fit(X_train)
    labels = m.predict(X_all)
    results[c] = (labels == -1).sum()
    print(f"contamination={c:.2f}  →  {results[c]} anomalies flagged")

## 6. Incident Overlay

In [ ]:
with open(INCIDENTS_FILE) as f:
    raw_incidents = json.load(f)

# Normalise to list
incidents = raw_incidents if isinstance(raw_incidents, list) else raw_incidents.get("incidents", [])
print(f"{len(incidents)} incident(s) loaded")
for inc in incidents:
    print(f"  {inc['start']}  →  {inc['end']}  [{inc['type']}]")

In [ ]:
def shade_incidents(ax, incidents, alpha=0.15):
    colors = {"errors": "tomato", "slow": "orange"}
    handles = []
    for inc in incidents:
        s = pd.Timestamp(inc["start"], tz="UTC")
        e = pd.Timestamp(inc["end"],   tz="UTC")
        c = colors.get(inc["type"], "gray")
        ax.axvspan(s, e, color=c, alpha=alpha)
        handles.append(mpatches.Patch(color=c, alpha=0.4, label=f"incident: {inc['type']}"))
    return handles

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

for ax, col, color in zip(axes,
    ["log_count", "error_kw_sum", "anomaly_score"],
    ["steelblue", "tomato", "purple"]):
    ax.plot(windowed.index, windowed[col], color=color, linewidth=0.9)
    ax.scatter(anom_idx, windowed.loc[anom_idx, col], color="red", s=20, zorder=5)
    ax.set_ylabel(col)
    handles = shade_incidents(ax, incidents)

axes[0].set_title("Logs: anomaly detection vs ground-truth incidents")
axes[2].axhline(0, color="gray", linestyle="--", linewidth=0.7)
if handles:
    axes[0].legend(handles=list({h.get_label(): h for h in handles}.values()), fontsize=8)
plt.tight_layout()
plt.show()

## 7. Detection Summary Table

In [ ]:
rows = []
for inc in incidents:
    s = pd.Timestamp(inc["start"], tz="UTC")
    e = pd.Timestamp(inc["end"],   tz="UTC")
    in_window = windowed.loc[s:e]
    if in_window.empty:
        hit, ttd = False, None
    else:
        first_det = in_window[in_window["is_anomaly"] == 1]
        hit = not first_det.empty
        ttd = (first_det.index[0] - s).total_seconds() if hit else None
    rows.append({
        "type": inc["type"],
        "start": inc["start"],
        "detected": hit,
        "ttd_seconds": ttd,
        "window_rows": len(in_window),
    })

summary = pd.DataFrame(rows)
print(f"Detection rate: {summary['detected'].mean():.0%}")
display(summary)

In [ ]:
# False positive rate (anomalies outside incident windows)
def in_any_incident(ts, incidents):
    for inc in incidents:
        s = pd.Timestamp(inc["start"], tz="UTC")
        e = pd.Timestamp(inc["end"],   tz="UTC")
        if s <= ts <= e:
            return True
    return False

anomaly_times = windowed[windowed["is_anomaly"] == 1].index
fps = [t for t in anomaly_times if not in_any_incident(t, incidents)]
print(f"Anomalies flagged : {len(anomaly_times)}")
print(f"False positives   : {len(fps)}")
print(f"True positives    : {len(anomaly_times) - len(fps)}")